# Práctica 1: Segmentación de Clientes mediante RFM y K-Means

## 1. Contexto
En esta práctica trabajaremos con el dataset público **Online Retail** del *UCI Machine Learning Repository* (DOI: `10.24432/C5BW33`, Licencia CC BY 4.0). Contiene **541.909 líneas de transacción** registradas entre diciembre de 2010 y diciembre de 2011 por una empresa británica de comercio electrónico.

**Variables principales:**
- `InvoiceNo`: identificador de factura (los valores que empiezan por `C` indican cancelaciones).
- `Quantity`: unidades adquiridas.
- `InvoiceDate`: fecha y hora de la transacción.
- `UnitPrice`: precio unitario.
- `CustomerID`: identificador del cliente.
- `Country`: país del cliente.

**Objetivo:** Transformar las transacciones en un dataset de clientes mediante variables **RFM** (*Recency, Frequency, Monetary*) y aplicar **K-Means** para identificar segmentos comercialmente interpretables.

### 1.1. Importación de librerías y carga del dataset

In [5]:
# Importación de librerías principales
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Configuración visual
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

In [6]:
# Carga del dataset Online Retail (desde memoria, archivo local o UCI Repository)
import importlib, subprocess, sys
try:
    import openpyxl
except ImportError:
    print('Instalando openpyxl en el entorno actual...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
    importlib.invalidate_caches()
    import openpyxl

UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
LOCAL_XLSX = 'Online Retail.xlsx'
LOCAL_CSV = 'Online_Retail.csv'

if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    print('El dataset ya está cargado en memoria.')
elif os.path.exists(LOCAL_CSV):
    print(f'Cargando desde archivo CSV local: {LOCAL_CSV}...')
    df = pd.read_csv(LOCAL_CSV, encoding='ISO-8859-1', parse_dates=['InvoiceDate'])
elif os.path.exists(LOCAL_XLSX):
    print(f'Cargando desde archivo Excel local: {LOCAL_XLSX}...')
    df = pd.read_excel(LOCAL_XLSX)
else:
    print('Descargando dataset desde UCI Machine Learning Repository...')
    df = pd.read_excel(UCI_URL)
    try:
        df.to_csv(LOCAL_CSV, index=False, encoding='ISO-8859-1')
        print(f'Copia guardada localmente en {LOCAL_CSV}')
    except Exception:
        pass

print(f'Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]} columnas')
df.head()

El dataset ya está cargado en memoria.
Dimensiones del dataset: 541,909 filas y 8 columnas


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [7]:
# Información general de tipos de datos y valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


### Descripción detallada de las columnas del dataset

A partir de la salida de `df.info()` y de la documentación oficial del dataset **Online Retail** (541.909 registros y 8 columnas), se detalla el significado, tipo de dato y particularidades de cada variable:

| Columna | Tipo de dato | Valores no nulos | Descripción y relevancia para el análisis RFM |
| :--- | :--- | :--- | :--- |
| **`InvoiceNo`** | `object` (alfanumérico) | 541.909 (sin nulos) | **Número de factura.** Identificador único de 6 dígitos asignado a cada transacción/pedido. Si el código comienza por la letra **`'C'`** (ej. `C536379`), indica una **cancelación o devolución**. Es clave para filtrar pedidos anulados y calcular la **Frecuencia (*Frequency*)** contando facturas únicas por cliente. |
| **`StockCode`** | `object` (alfanumérico) | 541.909 (sin nulos) | **Código de producto/artículo.** Identificador único de 5 dígitos asignado a cada producto distinto del catálogo (incluye algunos códigos especiales como `POST` de gastos de envío o `M` de ajustes manuales). |
| **`Description`** | `object` (texto) | 540.455 (~1.454 nulos) | **Descripción comercial del producto.** Nombre del artículo adquirido. Presenta algunos valores nulos en ajustes de inventario, pero no afecta directamente al cálculo de las métricas RFM. |
| **`Quantity`** | `int64` (entero) | 541.909 (sin nulos) | **Cantidad de unidades** de cada producto por línea de transacción. Toma valores negativos en devoluciones/cancelaciones o ajustes de stock, por lo que será necesario conservar únicamente `Quantity > 0` para analizar compras reales. |
| **`InvoiceDate`** | `datetime64[ns]` | 541.909 (sin nulos) | **Fecha y hora de la factura.** Día y hora exacta en que se generó la transacción (entre el 01/12/2010 y el 09/12/2011). Es imprescindible para calcular la **Recencia (*Recency*)**: días transcurridos desde la última compra de cada cliente hasta la fecha de referencia. |
| **`UnitPrice`** | `float64` (decimal) | 541.909 (sin nulos) | **Precio unitario** del producto en libras esterlinas (£). Multiplicado por `Quantity` (`Quantity * UnitPrice`) permite obtener el importe total de cada línea para calcular el **Valor Monetario (*Monetary*)** de cada cliente. Deben filtrarse valores `<= 0`. |
| **`CustomerID`** | `float64` (numérico) | 406.829 (~135.080 nulos) | **Identificador único del cliente** (número de 5 dígitos). Aproximadamente un **24,9 %** de las filas no tienen `CustomerID` (compras de usuarios no registrados o ventas sin ficha). Al ser una segmentación a nivel de cliente, será obligatorio **eliminar las filas sin `CustomerID`** y convertirlo a entero/texto. |
| **`Country`** | `object` (texto) | 541.909 (sin nulos) | **País de residencia** del cliente donde se realiza el pedido (mayoritariamente *United Kingdom*, seguido de otros países europeos y del resto del mundo). |

---
## 2. Exploración inicial
Analizamos el dataset transaccional antes de agregar la información por cliente, desglosando cada punto en una celda independiente.

### 2.1. Número de filas, columnas y periodo temporal cubierto

In [ ]:
# 2.1. Número de filas, columnas y periodo temporal cubierto
n_filas, n_columnas = df.shape
fecha_min = pd.to_datetime(df['InvoiceDate']).min()
fecha_max = pd.to_datetime(df['InvoiceDate']).max()
duracion = fecha_max - fecha_min

print(f'Número total de filas:    {n_filas:,}')
print(f'Número total de columnas: {n_columnas} -> {list(df.columns)}')
print(f'Fecha primera transacción: {fecha_min}')
print(f'Fecha última transacción:  {fecha_max}')
print(f'Periodo temporal cubierto: {duracion.days} días (~{duracion.days / 30.44:.1f} meses)')

### 2.2. Tipos de datos y conversión de `InvoiceDate`

In [ ]:
# 2.2. Tipos de datos y conversión explícita de InvoiceDate a datetime
print('--- Tipos de datos originales ---')
print(df.dtypes)

# Aseguramos que InvoiceDate sea datetime64 y que InvoiceNo / StockCode sean texto para evaluar prefijos como "C"
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip()

print('\n--- Tipos de datos tras la conversión ---')
print(df.dtypes)

### 2.3. Valores nulos, especialmente en `CustomerID`

In [ ]:
# 2.3. Estudio de valores nulos por columna (recuento y porcentaje)
nulos_abs = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100)

resumen_nulos = pd.DataFrame({
    'Valores Nulos': nulos_abs,
    'Porcentaje (%)': nulos_pct,
    'Valores No Nulos': df.notnull().sum()
}).sort_values(by='Valores Nulos', ascending=False)

print(f"Filas sin CustomerID: {nulos_abs['CustomerID']:,} ({nulos_pct['CustomerID']:.2f}% del total)")
print(f"Filas con CustomerID válido: {df['CustomerID'].notnull().sum():,} ({100 - nulos_pct['CustomerID']:.2f}% del total)")
resumen_nulos

### 2.4. Facturas canceladas y cantidades o precios no positivos

In [ ]:
# 2.4. Análisis de facturas canceladas (empiezan por 'C') y cantidades o precios <= 0
mask_canceladas = df['InvoiceNo'].str.startswith('C', na=False)
mask_qty_no_pos = df['Quantity'] <= 0
mask_price_no_pos = df['UnitPrice'] <= 0
mask_price_neg = df['UnitPrice'] < 0
mask_price_zero = df['UnitPrice'] == 0

print(f'1) Facturas canceladas (InvoiceNo empieza por "C"):')
print(f'   - Líneas de transacción canceladas: {mask_canceladas.sum():,} ({mask_canceladas.mean()*100:.2f}%)')
print(f'   - Número de facturas únicas canceladas: {df.loc[mask_canceladas, "InvoiceNo"].nunique():,}\n')

print(f'2) Cantidades no positivas (Quantity <= 0):')
print(f'   - Total de líneas con Quantity <= 0: {mask_qty_no_pos.sum():,} ({mask_qty_no_pos.mean()*100:.2f}%)')
print(f'   - De ellas, pertenecen a facturas canceladas ("C"): {(mask_qty_no_pos & mask_canceladas).sum():,}')
print(f'   - Sin prefijo "C" (ajustes/mermas de inventario sin CustomerID): {(mask_qty_no_pos & ~mask_canceladas).sum():,}\n')

print(f'3) Precios unitarios no positivos (UnitPrice <= 0):')
print(f'   - Total de líneas con UnitPrice <= 0: {mask_price_no_pos.sum():,} ({mask_price_no_pos.mean()*100:.2f}%)')
print(f'   - Líneas con UnitPrice == 0 (regalos/ajustes): {mask_price_zero.sum():,}')
print(f'   - Líneas con UnitPrice < 0 (deuda incobrable / Adjust bad debt): {mask_price_neg.sum():,}')

### 2.5. Número de clientes, facturas, productos y países distintos

In [ ]:
# 2.5. Cardinalidad de entidades únicas en el dataset bruto
n_clientes = df['CustomerID'].nunique(dropna=True)
n_facturas_total = df['InvoiceNo'].nunique()
n_facturas_validas = df.loc[~df['InvoiceNo'].str.startswith('C'), 'InvoiceNo'].nunique()
n_productos_codigo = df['StockCode'].nunique()
n_productos_desc = df['Description'].nunique(dropna=True)
n_paises = df['Country'].nunique()

print(f'Clientes distintos (CustomerID válidos): {n_clientes:,}')
print(f'Facturas distintas totales (InvoiceNo):  {n_facturas_total:,} (de las cuales {n_facturas_validas:,} no son cancelaciones)')
print(f'Productos distintos por StockCode:       {n_productos_codigo:,}')
print(f'Descripciones de producto distintas:     {n_productos_desc:,}')
print(f'Países distintos (Country):              {n_paises:,}\n')

print('Top 10 países por número de líneas de transacción:')
top_paises = df['Country'].value_counts().head(10).to_frame(name='Transacciones')
top_paises['Porcentaje (%)'] = (top_paises['Transacciones'] / len(df)) * 100
top_paises

### 2.6. Distribución del importe de las transacciones y posibles valores extremos

In [ ]:
# 2.6. Cálculo del importe por línea (Quantity * UnitPrice), estadísticos y detección de valores extremos
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print('--- Estadísticos descriptivos de Quantity, UnitPrice y TotalPrice (importe por línea) ---')
percent_list = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
display(df[['Quantity', 'UnitPrice', 'TotalPrice']].describe(percentiles=percent_list).T)

print('\n--- Top 5 transacciones con MAYOR importe positivo (outliers superiores) ---')
display(df.nlargest(5, 'TotalPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TotalPrice', 'CustomerID', 'Country']])

print('\n--- Top 5 transacciones con MENOR importe (cancelaciones / ajustes extremos) ---')
display(df.nsmallest(5, 'TotalPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TotalPrice', 'CustomerID', 'Country']])

# Visualización de la distribución y valores extremos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Boxplot de TotalPrice evidenciando los valores extremos
sns.boxplot(x=df['TotalPrice'], ax=axes[0], color='salmon')
axes[0].set_title('Boxplot del Importe por Línea (con valores extremos)')
axes[0].set_xlabel('Importe de la línea (£)')

# Gráfico 2: Histograma del importe acotado entre percentiles 1% y 99% (compras normales)
p01, p99 = df['TotalPrice'].quantile([0.01, 0.99])
sns.histplot(df.loc[df['TotalPrice'].between(p01, p99), 'TotalPrice'], bins=60, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title(f'Distribución del Importe por Línea (Percentiles 1% a 99%: [{p01:.1f}£, {p99:.1f}£])')
axes[1].set_xlabel('Importe de la línea (£)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 2.7. Pregunta de análisis: ¿Por qué cada fila del archivo representa una línea de factura y no un cliente completo?

**Respuesta:**

Cada fila del dataset original tiene una **granularidad de línea de pedido (*order line-item*)** porque registra la compra de **un producto concreto (`StockCode`) dentro de una factura determinada (`InvoiceNo`)** realizada por un cliente (`CustomerID`) en una fecha específica (`InvoiceDate`).

En un sistema transaccional de comercio electrónico (OLTP) existe una relación jerárquica **1 a muchos ($1:N$)** en dos niveles:
1. **Un cliente (`CustomerID`) puede realizar múltiples compras/facturas (`InvoiceNo`)** a lo largo del año (por ejemplo, comprar en diciembre, marzo y noviembre).
2. **Una misma factura (`InvoiceNo`) suele incluir varios artículos distintos (`StockCode`)** en el carrito de la compra, generando una fila por cada producto diferente adquirido en ese pedido.

Por tanto, un mismo `CustomerID` aparece repetido en decenas o cientos de filas. Si aplicáramos K-Means directamente sobre este archivo sin agregarlo antes, estaríamos agrupando **líneas de productos sueltos** en lugar de **clientes**, distorsionando completamente la segmentación. Por ello es imprescindible agregar primero las transacciones por `CustomerID` mediante las métricas **RFM** (*Recency, Frequency, Monetary*), de modo que cada fila de la nueva tabla represente exactamente a **un único cliente**.

In [ ]:
# Demostración práctica: ejemplo del cliente 17850 y la factura 536365
ejemplo_factura = df[df['InvoiceNo'] == '536365']
ejemplo_cliente = df[df['CustomerID'] == 17850]

print(f'La factura 536365 (de un único pedido) ocupa {len(ejemplo_factura)} filas (una por producto del carrito).')
print(f'El cliente 17850 aparece en {len(ejemplo_cliente)} filas repartidas en {ejemplo_cliente["InvoiceNo"].nunique()} facturas distintas.')